# ERP → Google Sheets Daily Automation

This notebook generates report DataFrames, builds a local `master_df`, creates a strict JSON-safe payload, and uploads one previous-day batch to Google Apps Script.

In [53]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    APPS_SCRIPT_URL,
    API_TOKEN,
    SPREADSHEET_ID,
    REPORT_TIMEZONE,
    get_previous_report_date,
    format_sheet_name,
    validate_config,
)
from src.master_builder import build_master_dataframe
from src.json_payload import build_payload
from src.google_uploader import (
    check_apps_script_health,
    upload_to_google_apps_script,
)

validate_config()

print("Project root    :", PROJECT_ROOT)
print("Report timezone :", REPORT_TIMEZONE)
print("Spreadsheet ID  :", SPREADSHEET_ID)
print("Apps Script URL :", APPS_SCRIPT_URL)


Project root    : c:\Users\Sujon\Desktop\GoogleSpreadSheetAutomation
Report timezone : Asia/Dhaka
Spreadsheet ID  : 1944VCMThyqliN_VNRyJggHEf6j3cVMUw-d__8j9i6I0
Apps Script URL : https://script.google.com/macros/s/AKfycbzT3Di2kguDbdfPaongnvyBuw0cp6rjcNJ-Z7pvyW7bG2ZhTAziF32u0IJg_WdOT09K/exec


## 1. Optional Apps Script health check

Run this during setup/troubleshooting. It does not write a daily report.

In [54]:
health = check_apps_script_health()
print(health)
if not health.get("success"):
    raise RuntimeError(health.get("error", "Apps Script health check failed"))


{'success': True, 'message': 'ERP Google Sheets Web App is running'}


## 2. Generate your real ERP reports

Replace the sample DataFrames below with your existing ERP report-generation code. Do not change the upload architecture.

## Data Process Start Here

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from bs4 import BeautifulSoup

### Less Production Report

In [56]:
import pandas as pd

# ---------------------------------------------------------
# 1. Get Previous Day
# ---------------------------------------------------------
previous_day = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)

# ---------------------------------------------------------
# 2. Format Previous Day as DDMMYYYY
# ---------------------------------------------------------
date_str = previous_day.strftime("%d%m%Y")

In [57]:
date_str

'25092026'

In [58]:
# ---------------------------------------------------------
# 3. Build Dynamic File Path
# ---------------------------------------------------------
file_path = fr"..\Data\SDEA_{date_str}.xlsx"

# ---------------------------------------------------------
# 4. Load Dataset
# ---------------------------------------------------------
sdea_df = pd.read_excel(file_path)

# Preview
sdea_df.head(5)

,SL#,Line,Buyer,Style,Work Day,SMV,IE Target 100%,Day Target Qty,Achieve Qty,Achieve (%),Today Efficiency (%),Monthly Efficiency (%),Previous Day Efficiency (%),Today Produce Minute,Today Availabe Minute,Monthly Produce Minute,Monthly Availabe Minute,Factory
0,1,Line 01,NEXT,Y63406 - GIRLS L/S T-SHIRT,4,5.80,375,375,370,98.67,38.60,45.10,46.24,15750.0,40800.0,363870.55,805425.0,MGL
1,2,Line 01,ECI,S27823HCHAMB - MENS L/S SHIRT,1,17.90,755,755,760,100.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MGL
2,3,Line 02,ECI,SS27224-86P - LADIES L/S SHIRT,1,18.50,711,711,540,75.95,44.52,46.70,62.32,14800.0,33240.0,328496.51,702090.0,MGL
3,4,Line 02,ECI,SS27 224-86S - LADIES L/S SHIRT,1,18.50,264,264,260,98.48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MGL
4,5,Line 03,ECI,S27105HBLANCO - LADIES L/S SHIRT,2,23.45,594,594,480,80.81,36.50,40.57,37.22,11256.0,30840.0,282834.50,691515.0,MGL


In [59]:
# ---------------------------------------------------------
# 1. Get Previous Day
# ---------------------------------------------------------
previous_day = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)

# Copy source DataFrame
report_df = sdea_df.copy()


# ---------------------------------------------------------
# 2. Filter Previous Day Data
# ---------------------------------------------------------
# Find a suitable date column
date_columns = [
    col
    for col in report_df.columns
    if any(
        word in str(col).lower()
        for word in ["date", "work date", "production date"]
    )
]

# If a date column exists, filter for previous day
if date_columns:
    date_col = date_columns[0]

    report_df[date_col] = pd.to_datetime(
        report_df[date_col],
        errors="coerce"
    )

    report_df = report_df[
        report_df[date_col].dt.normalize() == previous_day
    ].copy()


# ---------------------------------------------------------
# 3. Remove Invalid / Total Lines
# ---------------------------------------------------------
report_df = report_df[
    report_df["Factory"].notna() &
    report_df["Line"].notna() &
    (
        report_df["Line"]
        .astype(str)
        .str.strip()
        .str.lower() != "total"
    )
].copy()


# ---------------------------------------------------------
# 4. Convert Required Columns to Numeric
# ---------------------------------------------------------
report_df["Day Target Qty"] = pd.to_numeric(
    report_df["Day Target Qty"],
    errors="coerce"
)

report_df["Achieve Qty"] = pd.to_numeric(
    report_df["Achieve Qty"],
    errors="coerce"
)

report_df["Achieve (%)"] = pd.to_numeric(
    report_df["Achieve (%)"],
    errors="coerce"
)


# ---------------------------------------------------------
# 5. Filter Achieve (%) <= 70
# ---------------------------------------------------------
# Use the existing Achieve (%) value directly.
# No calculation or grouping is performed.
less_production = report_df[
    report_df["Achieve (%)"] <= 70
].copy()


# ---------------------------------------------------------
# 6. Add "Less Production" Column
# ---------------------------------------------------------
less_production.insert(
    0,
    "Less Production",
    ["Yesterday"] + [""] * (len(less_production) - 1)
)


# ---------------------------------------------------------
# 7. Rename Percentage Column
# ---------------------------------------------------------
less_production = less_production.rename(
    columns={
        "Achieve (%)": "Percentage"
    }
)


# ---------------------------------------------------------
# 8. Select Required Columns
# ---------------------------------------------------------
less_production = less_production[
    [
        "Less Production",
        "Factory",
        "Line",
        "Buyer",
        "Style",
        "Day Target Qty",
        "Achieve Qty",
        "Percentage"
    ]
].copy()


# ---------------------------------------------------------
# 9. Sort Data
# ---------------------------------------------------------
less_production = less_production.sort_values(
    ["Factory", "Line"]
).reset_index(drop=True)


# ---------------------------------------------------------
# 10. Format Percentage
# ---------------------------------------------------------
less_production["Percentage"] = (
    less_production["Percentage"]
    .round(2)
    .map(lambda x: f"{x:.2f}")
)


# ---------------------------------------------------------
# 11. Format Quantity Columns
# ---------------------------------------------------------
less_production["Day Target Qty"] = (
    less_production["Day Target Qty"]
    .apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "")
)

less_production["Achieve Qty"] = (
    less_production["Achieve Qty"]
    .apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "")
)


# ---------------------------------------------------------
# 12. Display Report
# ---------------------------------------------------------
display(
        f"Less Production — {previous_day:%d-%b-%Y}"
)

display(less_production)

'Less Production — 25-Sep-2026'

,Less Production,Factory,Line,Buyer,Style,Day Target Qty,Achieve Qty,Percentage
0,,MGL,Line 01,STREETONE,346497/3413355,582,400,68.73
1,,MGL,Line 02,US POLO,AW26-FM-SKT-002,364,120,32.97
2,Yesterday,MGL,Line 06,NEXT,Y63406 - GIRLS PANT,"1,500","1,000",66.67
3,,MGL,Line 07,ECI,S27105HRAYAF - LADIES LONG PANT,780,500,64.10
4,,MGL,Line 08,H&M,EQ LUKE COLLARLESS SHIRT,980,410,41.84
5,,MGL,Line 09,ECI,S27823HCHAMB - MENS L/S SHIRT,0,0,0.00
6,,MGNSL,Line 06,TEDDY S.P.A.,ABTD67420PFIO,240,160,66.67
7,,MGNSL,Line 13,TEDDY S.P.A.,ABTD67420PFIO,184,113,61.41


### Fabric MRR Pending Report

In [60]:
# ---------------------------------------------------------
# 3. Build Dynamic File Path
# ---------------------------------------------------------
file_path = fr"..\Data\MRR_Pending_Fabric_{date_str}.xlsx"

# ---------------------------------------------------------
# 4. Load Dataset
# ---------------------------------------------------------
fabric_mrr_pending_df = pd.read_excel(file_path)

# Preview
fabric_mrr_pending_df.head(5)


,SL#,Booking No,Supplier,Style No,Season,TOD Start,TOD End,Buyer,Fabric,PI NO,PI Date,PI Qty,BB LC No,Receive Qty,Merchant,Left Qty
0,1,MGNSL-FPO-26F0157,"CHANGSHU HJC CO.,LTD.",MFM_FW27_BER_0002,NaN,2026-11-20,2026-12-05,LOJAS RENNER,54% LYOCELL 46% COTTON (BCI) TWILL SOLID DYED ...,2026-ITS-HJC031,2026-09-15,4550.0,107926060251,0.0,Nipun,4550.0
1,2,MGNSL-FPO-26F0157,"CHANGSHU HJC CO.,LTD.",MFM_SS27_CAL_0058,NaN,2026-12-05,2026-12-05,LOJAS RENNER,54% LYOCELL 46% COTTON (BCI) TWILL SOLID DYED ...,2026-ITS-HJC031,2026-09-15,5950.0,107926060251,0.0,Nipun,5950.0
2,3,MGNSL-FPO-26F0154,"NANTONG SHUANGYING TEXTILE TECHNOLOGY CO.,LTD",MFM_SS26_MAC_0019B,FW27,2026-11-15,2026-11-15,LOJAS RENNER,55% LILEN 45% COTTON (BCI) Y/D SLUB CHAMBRAY W...,2026-ITS-NST008-C,2026-09-22,2520.0,107926060247,0.0,Nipun,2520.0
3,4,MGNSL-FPO-26L0091,Silk International Ltd.,R75002,SS27,2026-11-18,2026-12-15,NITEX PTE LTD.,"100% BCI COTTON, 154X114/60X60, 154x114/60x60",SIL/MGL/051/2026,2026-09-15,1599.0,1079260401062,0.0,Babu,1599.0
4,5,MGNSL-FPO-26L0091,Silk International Ltd.,R75002,SS27,2026-11-18,2026-12-15,NITEX PTE LTD.,95% cotton 5% spnadx 1x1 rib 280gsm,SIL/MGL/051/2026,2026-09-15,114.0,1079260401062,0.0,Babu,114.0


In [61]:
fabric_mrr_pending_df.columns.tolist()


['SL#',
 'Booking No',
 'Supplier',
 'Style No',
 'Season',
 'TOD Start',
 'TOD End',
 'Buyer',
 'Fabric',
 'PI NO',
 'PI Date',
 'PI Qty',
 'BB LC No',
 'Receive Qty',
 'Merchant',
 'Left Qty']

In [62]:
import pandas as pd
from datetime import date


# ---------------------------------------------------------
# 1. Normalize Column Names
# ---------------------------------------------------------
normalized_columns = {
    str(column).strip().casefold(): column
    for column in fabric_mrr_pending_df.columns
}


# ---------------------------------------------------------
# 2. Resolve Columns Dynamically
# ---------------------------------------------------------
def resolve_column(candidates, required=True):
    column = next(
        (
            normalized_columns[name.strip().casefold()]
            for name in candidates
            if name.strip().casefold() in normalized_columns
        ),
        None,
    )

    if required and column is None:
        available = ", ".join(str(name) for name in fabric_mrr_pending_df.columns)

        raise KeyError(
            f"Could not find a required column. "
            f"Tried: {', '.join(candidates)}. "
            f"Available columns: {available}"
        )

    return column


# ---------------------------------------------------------
# 3. Resolve Required / Optional Columns
# ---------------------------------------------------------
date_col = resolve_column([
    "Last TOD Date",
    "TOD End",
    "TOD Start"
])

style_col = resolve_column([
    "Style",
    "Style No",
    "Style No."
])

qty_col = resolve_column([
    "Qty",
    "Left Qty",
    "Less Qty",
    "Pending Qty",
    "Balance Qty"
])

buyer_col = resolve_column(
    ["Buyer"],
    required=False
)

merchant_col = resolve_column(
    ["Merchant"],
    required=False
)


# ---------------------------------------------------------
# 4. Prepare Working DataFrame
# ---------------------------------------------------------
today = pd.Timestamp(date.today())
end_date = today + pd.Timedelta(days=32)

work = fabric_mrr_pending_df.copy()

work["TOD"] = (
    pd.to_datetime(
        work[date_col],
        errors="coerce"
    )
    .dt.normalize()
)

work["Qty"] = (
    pd.to_numeric(
        work[qty_col],
        errors="coerce"
    )
    .fillna(0)
)

work["Buyer"] = (
    work[buyer_col]
    if buyer_col
    else ""
)

work["Merchant"] = (
    work[merchant_col]
    if merchant_col
    else ""
)


# ---------------------------------------------------------
# 5. Filter Today to Today + 32 Days
# ---------------------------------------------------------
work = work.loc[
    work["TOD"].between(
        today,
        end_date
    )
].copy()


# ---------------------------------------------------------
# 6. Create Days Category
# ---------------------------------------------------------
days_from_today = (
    work["TOD"] - today
).dt.days

work["Days"] = pd.cut(
    days_from_today,
    bins=[-1, 10, 20, 32],
    labels=[
        "10 Days",
        "20 Days",
        "30 Days"
    ]
)


# ---------------------------------------------------------
# 7. Combine Unique Names
# ---------------------------------------------------------
def unique_names(values):
    names = (
        pd.Series(values)
        .dropna()
        .astype(str)
        .str.strip()
    )

    names = names[names.ne("")]

    return ", ".join(
        pd.unique(names)
    )


# ---------------------------------------------------------
# 8. Create Main Report
# ---------------------------------------------------------
report = (
    work
    .groupby(
        [style_col, "TOD"],
        dropna=False,
        as_index=False
    )
    .agg(
        Buyer=("Buyer", unique_names),
        Qty=("Qty", "sum"),
        Merchant=("Merchant", unique_names),
        Days=("Days", "first"),
    )
    .rename(
        columns={
            style_col: "Style"
        }
    )
    [
        [
            "Days",
            "TOD",
            "Buyer",
            "Style",
            "Qty",
            "Merchant"
        ]
    ]
    .sort_values(
        ["TOD", "Style"]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 9. Add "Fabric Delay" as First Column
# ---------------------------------------------------------
report.insert(
    0,
    "Fabric Delay",
    ""
)

# Put "Yesterday" only in the first row
if not report.empty:
    report.loc[report.index[0], "Fabric Delay"] = "Yesterday"


# ---------------------------------------------------------
# 10. Final DataFrame
# ---------------------------------------------------------
fabric_delay_df = report


# ---------------------------------------------------------
# 11. Return / Display
# ---------------------------------------------------------
fabric_delay_df

,Fabric Delay,Days,TOD,Buyer,Style,Qty,Merchant
0,Yesterday,10 Days,2026-09-28,H&M,1257452,31696.00,Fakrul
1,,10 Days,2026-09-28,NITEX PTE LTD.,AE2337,19251.00,Babu
2,,10 Days,2026-09-30,NITEX PTE LTD.,AE2401,21300.00,Babu
3,,10 Days,2026-09-30,Fashion Fleet Clothing,Woman Top Bottom Set (Shirt+TrauserPaijama)(Set),11306.30,Sabbir
4,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,LISA STRETCH - SOLID,4214.79,Nipun
5,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,LISA STRETCH - STRIPE,1639.98,Nipun
6,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,MEDINA - SIGNAIPREIS STRIPE,7336.63,Nipun
7,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,Medina - Signalpreis uni,5379.00,Nipun
8,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,VERA NEW - NOS,2760.00,Nipun
9,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,Vera new - solid,552.00,Nipun


### Line Wise Earning Report

In [63]:
import pandas as pd

# ---------------------------------------------------------
# 3. Build Dynamic File Path
# ---------------------------------------------------------
file_path = fr"..\Data\DailySewingInfo_{date_str}.xlsx"

# ---------------------------------------------------------
# 4. Load Dataset
# ---------------------------------------------------------
DailySewingInfo = pd.read_excel(file_path)

# Preview
DailySewingInfo.head(5)

,Factory Name,Line No,Prod. Date,Working Hour,Buyer,Style Name,Style Code,SMV,FOB,CM,Order No,Body Part,Order Qty,Raw Qty,Target,Sewing Qty,Total CM,First TOD,Last TOD
0,MGL,Line-1,2026-09-23,NaN,EL CORTE INGLES,S27823HCHAMB,S27823HCHAMB,20.0,6.5000,1.9,0110990351301,NaN,9000,9000,NaN,760,1444.0,2026-10-15,2026-10-15
1,MGL,Line-1,2026-09-23,10.0,Geniemode Bd,Y63406,Y63406,22.0,6.8425,1.2,CJ8474434,NaN,7364,14728,NaN,370,444.0,2026-07-10,2026-07-10
2,MGL,Line-2,2026-09-23,NaN,EL CORTE INGLES,SS27 224-86P,SS27 224-86P,20.0,6.0000,2.1,0141591299001,NaN,450,450,NaN,40,84.0,2026-10-30,2026-10-30
3,MGL,Line-2,2026-09-23,NaN,EL CORTE INGLES,SS27 224-86P,SS27 224-86P,20.0,6.0000,2.1,0141591299001,NaN,450,450,NaN,500,1050.0,2026-10-30,2026-10-30
4,MGL,Line-2,2026-09-23,8.0,EL CORTE INGLES,SS27 224-86S,SS27 224-86S,20.0,6.0000,2.1,0141591299001,NaN,900,900,NaN,260,546.0,2026-10-30,2026-10-30


In [64]:
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================


FIXED_LINE_COST = 1925

FACTORIES = [
    "MGNSL",
    "MGL",
    "MGSL",
    "MGSL-DK",
]

LINE_WISE_TEXT = "*Quantity * CM - Fixed Line Cost Per Day USD $ 1925"


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

DailySewingInfo.columns = (
    DailySewingInfo.columns
      .astype(str)
      .str.strip()
)


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Factory Name",
    "Line No",
    "CM",
    "Sewing Qty"
]

missing_columns = [
    col for col in required_columns
    if col not in DailySewingInfo.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


# ============================================================
# 5. CLEAN REQUIRED DATA
# ============================================================

DailySewingInfo["Factory Name"] = (
    DailySewingInfo["Factory Name"]
    .astype(str)
    .str.strip()
)

DailySewingInfo["Line No"] = (
    DailySewingInfo["Line No"]
    .astype(str)
    .str.strip()
)

DailySewingInfo["CM"] = pd.to_numeric(
    DailySewingInfo["CM"],
    errors="coerce"
).fillna(0)

DailySewingInfo["Sewing Qty"] = pd.to_numeric(
    DailySewingInfo["Sewing Qty"],
    errors="coerce"
).fillna(0)


# Remove rows without Factory / Line
DailySewingInfo = DailySewingInfo[
    (DailySewingInfo["Factory Name"] != "") &
    (DailySewingInfo["Line No"] != "") &
    (DailySewingInfo["Factory Name"] != "nan") &
    (DailySewingInfo["Line No"] != "nan")
].copy()


# ============================================================
# 6. RECORD-WISE EARNING
# ============================================================
#
# Quantity * CM
#
# Example:
# 1100 * 1.20 = 1320
#
# ============================================================

DailySewingInfo["Record Earning"] = (
    DailySewingInfo["Sewing Qty"] * DailySewingInfo["CM"]
)


# ============================================================
# 7. GROUP FACTORY + LINE
# ============================================================
#
# Important:
#
# First calculate:
#
#     SUM(Quantity * CM)
#
# Then deduct:
#
#     1925
#
# ONLY ONCE per Factory + Line.
#
# ============================================================

grouped = (
    DailySewingInfo.groupby(
        ["Factory Name", "Line No"],
        as_index=False
    )["Record Earning"]
    .sum()
)


# ============================================================
# 8. DEDUCT FIXED LINE COST
# ============================================================

grouped["Earning"] = (
    grouped["Record Earning"]
    - FIXED_LINE_COST
)


# ============================================================
# 9. KEEP ONLY REQUIRED FACTORIES
# ============================================================

grouped = grouped[
    grouped["Factory Name"].isin(FACTORIES)
].copy()


# ============================================================
# 10. PIVOT FACTORY INTO COLUMNS
# ============================================================

pivot = grouped.pivot(
    index="Line No",
    columns="Factory Name",
    values="Earning"
)


# ============================================================
# 11. FORCE FACTORY COLUMN ORDER
# ============================================================

pivot = pivot.reindex(
    columns=FACTORIES
)


# ============================================================
# 12. DYNAMICALLY SORT LINE NUMBERS
# ============================================================

def line_sort_key(value):
    """
    Line-1
    Line-2
    Line-10

    will become:
    1, 2, 10
    instead of:
    1, 10, 2
    """

    value = str(value)

    try:
        return int(
            value.split("-")[-1]
        )
    except ValueError:
        return 999999


pivot = pivot.sort_index(
    key=lambda x: x.map(line_sort_key)
)


# ============================================================
# 13. RENAME FACTORY COLUMNS
# ============================================================

pivot.columns = [
    f"{factory} (Earning)"
    for factory in pivot.columns
]


# ============================================================
# 14. RESET INDEX
# ============================================================

df_summary = pivot.reset_index()


# ============================================================
# 15. ADD "Line wise Earning" BEFORE "Line No"
# ============================================================

df_summary.insert(
    0,
    "Line wise Earning",
    ""
)

# Put text only in first cell
if not df_summary.empty:
    df_summary.loc[
        df_summary.index[0],
        "Line wise Earning"
    ] = LINE_WISE_TEXT


# ============================================================
# 16. FINAL COLUMN ORDER
# ============================================================

line_wise_earning = df_summary[
    [
        "Line wise Earning",
        "Line No",
        "MGNSL (Earning)",
        "MGL (Earning)",
        "MGSL (Earning)",
        "MGSL-DK (Earning)",
    ]
]


# ============================================================
# 17. DISPLAY DATAFRAME
# ============================================================

line_wise_earning

,Line wise Earning,Line No,MGNSL (Earning),MGL (Earning),MGSL (Earning),MGSL-DK (Earning)
0,*Quantity * CM - Fixed Line Cost Per Day USD $...,Line-1,-1029.000,-37.0,-1225.000,-469.00
1,,Line-2,-53.000,-245.0,-1445.000,-568.34
2,,Line-3,-496.952,-485.0,-771.776,-518.88
3,,Line-4,-1177.500,-973.0,-21.000,-245.00
4,,Line-5,-1120.000,-815.0,-21.000,-435.50
5,,Line-6,-1739.400,-725.0,-21.000,NaN
6,,Line-7,-637.000,-425.0,35.000,NaN
7,,Line-8,-765.000,139.0,254.770,NaN
8,,Line-9,-53.000,-187.6,-21.000,NaN
9,,Line-10,-1189.000,51.0,NaN,NaN


### OT and EOT

In [65]:
# oteot = pd.read_excel("..\Data\CategoryWiseOverTime.xlsx", sheet_name="Report_CategoryWiseOTWoven")

In [66]:
# =============================================================================
# Category Wise Over Time Report
# Clean Report_CategoryWiseOTWoven -> return `oteot` DataFrame only
# =============================================================================

import re
import warnings
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

_MONTHS = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

# Excel 1900 date system
_EXCEL_EPOCH = pd.Timestamp("1899-12-30")

# Only whole-number values inside this range are treated as Excel dates
# 2000-01-01 -> 2100-12-31
_MIN_SERIAL = 36526
_MAX_SERIAL = 73415

# Expected final date format
_DATE_RE = re.compile(r"^\d{2}-[A-Z][a-z]{2}-\d{4}$")

# Header matching keys
_HOUR_KEYS = {"hr", "hour", "hours"}
_AMNT_KEYS = {"amnt", "amount"}


# -----------------------------------------------------------------------------
# Helper Functions
# -----------------------------------------------------------------------------

def _clean_cell(value):
    """
    Normalize one Excel cell.

    Rules:
    - Remove zero-width characters
    - Remove line breaks/tabs
    - Collapse multiple spaces
    - Trim leading/trailing spaces
    - Convert blank strings / None / NaN to np.nan
    - Keep numbers and dates unchanged
    """

    if isinstance(value, str):
        value = re.sub(r"[\u200b\ufeff]", "", value)
        value = re.sub(r"\s+", " ", value).strip()

        return np.nan if value == "" else value

    if value is None or pd.isna(value):
        return np.nan

    return value


def _key(value):
    """
    Convert header text into a comparison key.

    Examples:
        'Hr.'          -> 'hr'
        'Hour'         -> 'hour'
        'Amnt.'        -> 'amnt'
        'Month Total'  -> 'monthtotal'
        'Factory'      -> 'factory'
    """

    return re.sub(r"[^a-z]", "", str(value).lower())


def _fmt_date(value):
    """
    Convert a date-like value into dd-MMM-yyyy format.
    """

    return (
        f"{value.day:02d}-"
        f"{_MONTHS[value.month - 1]}-"
        f"{value.year}"
    )


def _to_date_label(value):
    """
    Convert:
        - datetime/date/Timestamp
        - Excel serial date

    into:
        dd-MMM-yyyy

    Return None if the value is not a valid date.
    """

    # Real Python/Pandas date
    if isinstance(value, (datetime, date, pd.Timestamp)):
        return _fmt_date(value)

    # Excel serial number
    if (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, bool)
    ):
        numeric_value = float(value)

        if (
            numeric_value.is_integer()
            and _MIN_SERIAL <= numeric_value <= _MAX_SERIAL
        ):
            converted_date = (
                _EXCEL_EPOCH
                + pd.Timedelta(days=int(numeric_value))
            )

            return _fmt_date(converted_date)

    return None


def _ffill(values):
    """
    Forward-fill values without relying on pandas downcasting.
    """

    output = []
    last_value = np.nan

    for value in values:

        if pd.isna(value):
            output.append(last_value)

        else:
            last_value = value
            output.append(value)

    return output


# -----------------------------------------------------------------------------
# Main Function
# -----------------------------------------------------------------------------

def clean_category_wise_overtime(
    input_file,
    source_sheet="Report_CategoryWiseOTWoven",
    rows_to_drop=11,
    fill_columns=("Factory", "Type", "Category"),
):
    """
    Read and clean the Category Wise Over Time Excel report.

    Returns
    -------
    pandas.DataFrame
        Cleaned `oteot` DataFrame.

    Final structure
    ---------------

    Row 0:
        Month Total | ... | 23-Sep-2026 | ... | 24-Sep-2026 | ...

    Row 1:
        Factory | Type | Category | Type | Emp | Hour | Amnt. | Emp | Hr. | Amnt. | ...

    Row 2+:
        Actual OT/EOT data

    No Excel file is created or saved.
    """

    # =========================================================================
    # 1. Validate input file
    # =========================================================================

    input_file = Path(input_file)

    if not input_file.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_file}"
        )

    # =========================================================================
    # 2. Read source worksheet
    # =========================================================================

    with pd.ExcelFile(input_file) as excel:

        if source_sheet not in excel.sheet_names:
            raise ValueError(
                f"Sheet '{source_sheet}' not found.\n"
                f"Available sheets: {excel.sheet_names}"
            )

        raw = excel.parse(
            source_sheet,
            header=None,
            dtype=object,
            keep_default_na=False
        )

    # =========================================================================
    # 3. Remove report information rows
    #
    # Example:
    # rows_to_drop = 11
    #
    # iloc[11:] means:
    # Excel row 12 onward
    # =========================================================================

    df = (
        raw
        .iloc[rows_to_drop:]
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"The sheet has no rows after removing "
            f"the first {rows_to_drop} rows."
        )

    # Use zero-based integer column indexes
    df.columns = range(df.shape[1])

    # =========================================================================
    # 4. Normalize every cell
    # =========================================================================

    elementwise = (
        df.map
        if hasattr(df, "map")
        else df.applymap
    )

    df = elementwise(_clean_cell).astype(object)

    # =========================================================================
    # 5. Remove completely blank rows
    # =========================================================================

    df = (
        df
        .dropna(how="all")
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            "The worksheet became empty after removing blank rows."
        )

    # =========================================================================
    # 6. Convert Excel serial dates in first row
    #
    # Example:
    # 46288 -> 23-Sep-2026
    # =========================================================================

    for column in df.columns:

        date_label = _to_date_label(
            df.iat[0, column]
        )

        if date_label:
            df.iat[0, column] = date_label

    # =========================================================================
    # 7. Locate actual header row
    #
    # Expected:
    #
    # Row 0 -> Month Total / Dates
    # Row 1 -> Factory / Type / Category / Emp / Hr. / Amnt.
    # =========================================================================

    header_rows = [
        row_index
        for row_index in df.index[:5]
        if any(
            _key(value) == "factory"
            for value in df.loc[row_index].dropna()
        )
    ]

    if header_rows != [1]:

        raise ValueError(
            "Expected the date row at position 0 and the "
            "'Factory ... Emp/Hr./Amnt.' row at position 1 "
            f"after removing {rows_to_drop} rows.\n"
            f"Found header row(s): {header_rows}.\n"
            "Please check `rows_to_drop`."
        )

    # =========================================================================
    # 8. Create header lookup
    # =========================================================================

    hdr = {
        column: _key(value)
        for column, value in df.loc[1].items()
        if pd.notna(value)
    }

    # =========================================================================
    # 9. Find all Emp columns
    #
    # Every OT/EOT group is:
    #
    # Emp | Hr. | Amnt.
    #
    # Example:
    #
    # Emp | Hr. | Amnt. | Emp | Hr. | Amnt.
    # =========================================================================

    emp_cols = sorted(
        column
        for column, header_key in hdr.items()
        if header_key == "emp"
    )

    if not emp_cols:
        raise ValueError(
            "No 'Emp' columns found in the header row."
        )

    # =========================================================================
    # 10. Detect every Emp/Hr./Amnt. group
    # =========================================================================

    bounds = emp_cols[1:] + [df.shape[1]]

    groups = []

    for emp_column, next_column in zip(
        emp_cols,
        bounds
    ):

        # Find Hour/Hr. column
        hr_column = next(
            (
                column
                for column in range(
                    emp_column + 1,
                    next_column
                )
                if hdr.get(column) in _HOUR_KEYS
            ),
            None
        )

        # Find Amnt./Amount column
        amount_column = next(
            (
                column
                for column in range(
                    emp_column + 1,
                    next_column
                )
                if (
                    hdr.get(column) in _AMNT_KEYS
                    and hr_column is not None
                    and column > hr_column
                )
            ),
            None
        )

        if hr_column is None or amount_column is None:

            raise ValueError(
                "Column group starting at column index "
                f"{emp_column} does not contain a valid "
                "Emp / Hr. / Amnt. structure."
            )

        groups.append(
            {
                "emp": emp_column,
                "hr": hr_column,
                "amnt": amount_column,
                "label": None,
                "kind": None,
            }
        )

    # =========================================================================
    # 11. Read Month Total / Date labels from first row
    # =========================================================================

    labels = []

    for column, value in df.loc[0].items():

        if pd.isna(value):
            continue

        value_key = _key(value)

        # Month Total
        if value_key == "monthtotal":

            labels.append(
                (
                    column,
                    "Month Total",
                    "month_total"
                )
            )

        # Date
        elif (
            isinstance(value, str)
            and _DATE_RE.match(value)
        ):

            labels.append(
                (
                    column,
                    value,
                    "date"
                )
            )

    # =========================================================================
    # 12. Validate date headers
    # =========================================================================

    if not any(
        kind == "date"
        for _, _, kind in labels
    ):

        raise ValueError(
            "No date headers found in the first row."
        )

    if not any(
        kind == "month_total"
        for _, _, kind in labels
    ):

        warnings.warn(
            "'Month Total' was not found in the first row."
        )

    # =========================================================================
    # 13. Attach Month Total / Date labels to Emp groups
    # =========================================================================

    for column, text, kind in labels:

        # Find first Emp group whose Emp column is at or
        # to the right of the label
        group = next(
            (
                group
                for group in groups
                if group["emp"] >= column
            ),
            None
        )

        if group is None:

            raise ValueError(
                f"Header '{text}' at column index "
                f"{column} cannot be matched to an "
                "Emp/Hr./Amnt. group."
            )

        if group["label"] is not None:

            raise ValueError(
                f"Header '{text}' is being assigned to an "
                "Emp/Hr./Amnt. group that already has "
                f"the label '{group['label']}'."
            )

        # Ensure Hr. and Amnt. cells are blank
        if any(
            pd.notna(df.iat[0, column_index])
            for column_index in (
                group["hr"],
                group["amnt"]
            )
        ):

            raise ValueError(
                f"Unexpected content found after "
                f"'{text}'. Refusing to overwrite data."
            )

        group["label"] = text
        group["kind"] = kind

        # Move label to Emp column if necessary
        if column != group["emp"]:

            df.iat[0, column] = np.nan
            df.iat[0, group["emp"]] = text

    # =========================================================================
    # 14. Check groups without labels
    # =========================================================================

    for group in groups:

        if group["label"] is None:

            warnings.warn(
                "Emp/Hr./Amnt. group at column index "
                f"{group['emp']} has no date header."
            )

    # =========================================================================
    # 15. Forward-fill Factory / Type / Category
    #
    # IMPORTANT:
    # Only the FIRST matching header is selected.
    #
    # This prevents the second 'Type' column, which represents
    # OT/EOT type, from being incorrectly forward-filled.
    # =========================================================================

    fill_cols = []

    for name in fill_columns:

        target_key = _key(name)

        column = next(
            (
                column
                for column, header_key in sorted(hdr.items())
                if header_key == target_key
            ),
            None
        )

        if column is None:

            raise ValueError(
                f"Could not find '{name}' "
                "header for forward-fill."
            )

        fill_cols.append(column)

    # Only actual data rows: row 2 onward
    for column in fill_cols:

        df.loc[2:, column] = _ffill(
            df.loc[2:, column].tolist()
        )

    # =========================================================================
    # 16. Remove completely blank columns
    #
    # These are normally leftovers from merged Excel cells.
    # =========================================================================

    keep_columns = [
        column
        for column in df.columns
        if df[column].notna().any()
    ]

    if not keep_columns:

        raise ValueError(
            "No non-blank columns remain after cleaning."
        )

    # Map original column positions -> new positions
    new_position = {
        old_column: new_column
        for new_column, old_column
        in enumerate(keep_columns)
    }

    # =========================================================================
    # 17. Create final `oteot` DataFrame
    # =========================================================================

    oteot = df[keep_columns].copy()

    # Reset columns to zero-based integers
    oteot.columns = range(
        len(keep_columns)
    )

    # =========================================================================
    # 18. Update group positions after blank-column removal
    # =========================================================================

    for group in groups:

        try:
            group["emp"] = new_position[
                group["emp"]
            ]

            group["hr"] = new_position[
                group["hr"]
            ]

            group["amnt"] = new_position[
                group["amnt"]
            ]

        except KeyError as exc:

            raise ValueError(
                "A required Emp/Hr./Amnt. column "
                "was removed unexpectedly during "
                "blank-column cleanup."
            ) from exc

        # Every group must remain:
        #
        # Emp | Hr. | Amnt.
        #
        if not (
            group["hr"] == group["emp"] + 1
            and
            group["amnt"] == group["emp"] + 2
        ):

            raise ValueError(
                f"Group '{group['label']}' is not arranged "
                "as three adjacent columns after cleaning."
            )

    # =========================================================================
    # 19. Final DataFrame cleanup
    # =========================================================================

    oteot = oteot.reset_index(drop=True)

    # =========================================================================
    # 20. Return ONLY the DataFrame
    # =========================================================================

    return oteot


In [67]:
# # =============================================================================
# # oteot -> OT / Extra-OT Management Report
# # Returns pandas DataFrame only
# # No Excel export / No file download
# # =============================================================================

# import re
# import numpy as np
# import pandas as pd


# # =============================================================================
# # Fixed zero-based column positions inside `oteot`
# #
# # Row 0 = Date / reporting-period information
# # Row 1 = Actual column headers
# # Row 2 onward = Data
# # =============================================================================

# COLS = {
#     "Factory": 0,
#     "Type": 1,
#     "Category": 2,
#     "OTEOT": 3,

#     # Month Total
#     "MT_Emp": 4,
#     "MT_Hour": 5,
#     "MT_Amnt": 6,

#     # Previous Day
#     "PD_Emp": 7,
#     "PD_Hour": 8,
#     "PD_Amnt": 9,
# }


# NUMERIC_COLS = [
#     "MT_Emp",
#     "MT_Hour",
#     "MT_Amnt",
#     "PD_Emp",
#     "PD_Hour",
#     "PD_Amnt",
# ]


# # =============================================================================
# # KPI definitions
# # =============================================================================

# KPI_SPEC = [
#     (
#         "Number of OT Employees",
#         "PD_Emp",
#         "OT",
#     ),

#     (
#         "Number of Extra OT Employees",
#         "PD_Emp",
#         "EOT",
#     ),

#     (
#         "Previous Days' OT Hours",
#         "PD_Hour",
#         "OT",
#     ),

#     (
#         "Previous Days' Extra OT Hours",
#         "PD_Hour",
#         "EOT",
#     ),

#     (
#         "Monthly Cumulative OT Hours",
#         "MT_Hour",
#         "OT",
#     ),

#     (
#         "Monthly Cumultiave Extra OT Hours",
#         "MT_Hour",
#         "EOT",
#     ),

#     (
#         "Monthly OT Expense",
#         "MT_Amnt",
#         "OT",
#     ),

#     (
#         "Monthly Extra OT Expense",
#         "MT_Amnt",
#         "EOT",
#     ),
# ]


# REPORT_KPI_COLS = [
#     "Number of OT Employees",
#     "Number of Extra OT Employees",
#     "Previous Days' OT Hours",
#     "Previous Days' Extra OT Hours",
#     "Monthly Cumulative OT Hours",
#     "Monthly Cumultiave Extra OT Hours",
#     "Monthly OT Expense",
#     "Monthly Extra OT Expense",
# ]


# # =============================================================================
# # Source subtotal / total rows
# # =============================================================================

# _AGG_CATEGORY_KEYS = {
#     "total",
#     "factot",
#     "facteot",
# }

# _AGG_FACTORY_KEYS = {
#     "totalot",
#     "totaleot",
#     "grandtotal",
# }


# # =============================================================================
# # Helper: normalize text for comparisons
# # =============================================================================

# def _norm_key(value):
#     """
#     Convert a value into a normalized alphabetic key.

#     Examples
#     --------
#     'Fact OT'    -> 'factot'
#     'Grand Total' -> 'grandtotal'
#     'Woven'      -> 'woven'
#     """
#     return re.sub(
#         r"[^a-z]",
#         "",
#         str(value).lower()
#     )


# # =============================================================================
# # Helper: clean text values
# # =============================================================================

# def _clean_text(value):
#     """
#     Normalize whitespace and safely convert text values.
#     """
#     if pd.isna(value):
#         return np.nan

#     value = str(value)

#     value = re.sub(
#         r"\s+",
#         " ",
#         value
#     ).strip()

#     return value if value else np.nan


# # =============================================================================
# # Main report function
# # =============================================================================

# def build_ot_management_report(oteot):
#     """
#     Build OT / Extra-OT management report from cleaned `oteot`.

#     Parameters
#     ----------
#     oteot : pandas.DataFrame
#         Cleaned Category Wise OT report.

#     Returns
#     -------
#     management_report : pandas.DataFrame
#         Factory-wise management report containing exactly 8 KPI columns.

#     report_info : dict
#         Reporting period information.

#     detail_kpi : pandas.DataFrame
#         Detailed Factory + Type + Category KPI calculation.
#     """

#     # =========================================================================
#     # 1. Validate input
#     # =========================================================================

#     if not isinstance(oteot, pd.DataFrame):
#         raise TypeError(
#             "`oteot` must be a pandas DataFrame."
#         )

#     required_rows = 3
#     required_columns = max(COLS.values()) + 1

#     if oteot.shape[0] < required_rows:
#         raise ValueError(
#             "`oteot` must contain at least 3 rows."
#         )

#     if oteot.shape[1] < required_columns:
#         raise ValueError(
#             f"`oteot` must contain at least "
#             f"{required_columns} columns."
#         )

#     # =========================================================================
#     # 2. Read reporting-period information from row 0
#     # =========================================================================

#     date_labels = []

#     for value in oteot.iloc[0]:

#         if isinstance(value, str):

#             value = value.strip()

#             if re.match(
#                 r"^\d{2}-[A-Za-z]{3}-\d{4}$",
#                 value
#             ):
#                 date_labels.append(value)

#     report_info = {}

#     if date_labels:

#         parsed_dates = pd.to_datetime(
#             date_labels,
#             format="%d-%b-%Y",
#             errors="coerce"
#         )

#         parsed_dates = parsed_dates[
#             ~parsed_dates.isna()
#         ]

#         if len(parsed_dates) > 0:

#             parsed_dates = sorted(parsed_dates)

#             report_info["Previous Day"] = (
#                 parsed_dates[-1].strftime("%d-%b-%Y")
#             )

#             report_info["Month Total Period"] = (
#                 f"{parsed_dates[0]:%d-%b-%Y} "
#                 f"to "
#                 f"{parsed_dates[-1]:%d-%b-%Y}"
#             )

#     # =========================================================================
#     # 3. Extract actual data
#     #
#     # Row 0 -> reporting information
#     # Row 1 -> headers
#     # Row 2+ -> data
#     # =========================================================================

#     data = oteot.iloc[2:].copy()

#     data = data.iloc[
#         :,
#         list(COLS.values())
#     ].copy()

#     data.columns = list(COLS.keys())

#     # =========================================================================
#     # 4. Clean text columns
#     # =========================================================================

#     TEXT_COLS = [
#         "Factory",
#         "Type",
#         "Category",
#         "OTEOT",
#     ]

#     for column in TEXT_COLS:

#         data[column] = data[column].apply(
#             _clean_text
#         )

#     # =========================================================================
#     # 5. Remove completely invalid rows
#     # =========================================================================

#     data = data.dropna(
#         subset=[
#             "Factory",
#             "Category",
#             "OTEOT",
#         ]
#     ).reset_index(drop=True)

#     # =========================================================================
#     # 6. Convert numeric columns
#     # =========================================================================

#     for column in NUMERIC_COLS:

#         data[column] = pd.to_numeric(
#             data[column],
#             errors="coerce"
#         ).fillna(0)

#     # =========================================================================
#     # 7. Remove source subtotal / total rows
#     # =========================================================================

#     category_key = data["Category"].apply(
#         _norm_key
#     )

#     factory_key = data["Factory"].apply(
#         _norm_key
#     )

#     is_aggregate = (
#         category_key.isin(_AGG_CATEGORY_KEYS)
#         |
#         factory_key.isin(_AGG_FACTORY_KEYS)
#     )

#     real = data.loc[
#         ~is_aggregate
#     ].reset_index(drop=True)

#     if real.empty:

#         raise ValueError(
#             "No valid OT / Extra-OT category rows "
#             "remained after removing subtotal and "
#             "grand-total rows."
#         )

#     # =========================================================================
#     # 8. Aggregate Factory + Type + Category + OT/EOT
#     # =========================================================================

#     grouped = (
#         real
#         .groupby(
#             [
#                 "Factory",
#                 "Type",
#                 "Category",
#                 "OTEOT",
#             ],
#             sort=False,
#             dropna=False,
#         )[NUMERIC_COLS]
#         .sum()
#         .reset_index()
#     )

#     # =========================================================================
#     # 9. Create OT / EOT pivot
#     # =========================================================================

#     pivot = grouped.pivot_table(
#         index=[
#             "Factory",
#             "Type",
#             "Category",
#         ],

#         columns="OTEOT",

#         values=NUMERIC_COLS,

#         fill_value=0,

#         aggfunc="sum",
#     )

#     # =========================================================================
#     # 10. Calculate detailed KPI
#     # =========================================================================

#     detail_kpi = pd.DataFrame(
#         index=pivot.index
#     )

#     for kpi_name, source_column, ot_eot in KPI_SPEC:

#         column_key = (
#             source_column,
#             ot_eot
#         )

#         if column_key in pivot.columns:

#             detail_kpi[kpi_name] = (
#                 pivot[column_key]
#             )

#         else:

#             detail_kpi[kpi_name] = 0

#     detail_kpi = (
#         detail_kpi
#         .reset_index()
#     )

#     # =========================================================================
#     # 11. Ensure numeric KPI columns
#     # =========================================================================

#     for column in REPORT_KPI_COLS:

#         detail_kpi[column] = pd.to_numeric(
#             detail_kpi[column],
#             errors="coerce"
#         ).fillna(0)

#     # =========================================================================
#     # 12. Factory + Type summary
#     # =========================================================================

#     type_summary = (
#         detail_kpi
#         .groupby(
#             [
#                 "Factory",
#                 "Type",
#             ],
#             sort=False,
#             dropna=False,
#         )[REPORT_KPI_COLS]
#         .sum()
#         .reset_index()
#     )

#     # =========================================================================
#     # 13. Build management report
#     #
#     # Logic:
#     #
#     # If a factory has Woven:
#     #
#     #     MGL
#     #     MGL-Knit
#     #
#     # instead of:
#     #
#     #     MGL-Woven
#     #     MGL-Knit
#     #
#     # If a factory has only one Type:
#     #
#     #     Factory
#     #
#     # is used directly.
#     # =========================================================================

#     rows = []

#     BASE_TYPE_KEY = "woven"

#     factory_order = list(
#         dict.fromkeys(
#             real["Factory"].tolist()
#         )
#     )

#     for factory in factory_order:

#         factory_types = (
#             type_summary.loc[
#                 type_summary["Factory"] == factory,
#                 "Type"
#             ]
#             .tolist()
#         )

#         if not factory_types:
#             continue

#         # ---------------------------------------------------------------------
#         # Find Woven type
#         # ---------------------------------------------------------------------

#         woven_type = next(
#             (
#                 item
#                 for item in factory_types
#                 if _norm_key(item) == BASE_TYPE_KEY
#             ),
#             None
#         )

#         # ---------------------------------------------------------------------
#         # Determine factory/base row
#         # ---------------------------------------------------------------------

#         if woven_type is not None:

#             base_type = woven_type

#         elif len(factory_types) == 1:

#             base_type = factory_types[0]

#         else:

#             base_type = factory_types[0]

#         # ---------------------------------------------------------------------
#         # Get base type summary
#         # ---------------------------------------------------------------------

#         base_row = type_summary.loc[
#             (
#                 type_summary["Factory"] == factory
#             )
#             &
#             (
#                 type_summary["Type"] == base_type
#             )
#         ]

#         if base_row.empty:
#             continue

#         base_values = (
#             base_row.iloc[0]
#             [REPORT_KPI_COLS]
#             .to_dict()
#         )

#         rows.append(
#             {
#                 "Factory": factory,
#                 **base_values,
#             }
#         )

#         # ---------------------------------------------------------------------
#         # Add other types
#         # ---------------------------------------------------------------------

#         for extra_type in factory_types:

#             if extra_type == base_type:
#                 continue

#             extra_row = type_summary.loc[
#                 (
#                     type_summary["Factory"] == factory
#                 )
#                 &
#                 (
#                     type_summary["Type"] == extra_type
#                 )
#             ]

#             if extra_row.empty:
#                 continue

#             extra_values = (
#                 extra_row.iloc[0]
#                 [REPORT_KPI_COLS]
#                 .to_dict()
#             )

#             rows.append(
#                 {
#                     "Factory": f"{factory}-{extra_type}",
#                     **extra_values,
#                 }
#             )

#     # =========================================================================
#     # 14. Grand Total
#     # =========================================================================

#     total_row = {
#         "Factory": "Total"
#     }

#     for column in REPORT_KPI_COLS:

#         total_row[column] = (
#             detail_kpi[column].sum()
#         )

#     rows.append(total_row)

#     # =========================================================================
#     # 15. Create final management report
#     # =========================================================================

#     management_report = pd.DataFrame(
#         rows,
#         columns=[
#             "Factory",
#             *REPORT_KPI_COLS,
#         ]
#     )

#     # =========================================================================
#     # 16. Format numeric values
#     # =========================================================================

#     for column in REPORT_KPI_COLS:

#         management_report[column] = pd.to_numeric(
#             management_report[column],
#             errors="coerce"
#         )

#     # =========================================================================
#     # 17. Return only data — no Excel creation
#     # =========================================================================

#     return (
#         management_report,
#         report_info,
#         detail_kpi,
#     )


# # =============================================================================
# # Run
# # =============================================================================
# #
# # `clean_category_wise_overtime()` should already be available from your
# # cleaning cell.
# # =============================================================================

# oteot = clean_category_wise_overtime(
#     input_file=r"..\Data\CategoryWiseOverTime.xlsx",
#     source_sheet="Report_CategoryWiseOTWoven",
# )


# # =============================================================================
# # Build report
# # =============================================================================

# oteot_report, report_info, detail_kpi = (
#     build_ot_management_report(oteot)
# )


# # =============================================================================
# # Final report
# # =============================================================================

# display(
#     oteot_report
# )

In [68]:
# =============================================================================
# OT / EOT MANAGEMENT REPORT
# =============================================================================
# Purpose:
#   Build a Factory-wise OT / Extra-OT management report from the cleaned
#   `oteot` DataFrame.
#
# Input:
#   ..\Data\CategoryWiseOverTime_{date_str}.xlsx
#
# Output DataFrame:
#   OT and EOT
#   Factory
#   Total Employees Present
#   Number of OT Employees
#   Number of Extra OT Employees
#   Previous Days' OT Hours
#   Previous Days' Extra OT Hours
#   Monthly Cumulative OT Hours
#   Monthly Cumultiave Extra OT Hours
#   Monthly OT Expense
#   Monthly Extra OT Expense
# =============================================================================

import re
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# 1. INPUT FILE CONFIGURATION
# =============================================================================

# -------------------------------------------------------------------------
# Date configuration
# -------------------------------------------------------------------------
# By default, use the previous calendar day.
#
# Example:
#   If today = 26-Sep-2026
#   date_str = "25_Sep_2026"
#
# To use a specific date manually, replace None with:
#
#   MANUAL_DATE_STR = "25_Sep_2026"
# -------------------------------------------------------------------------


# -------------------------------------------------------------------------
# Dynamic input filename
# -------------------------------------------------------------------------

INPUT_FILE = Path(
    rf"..\Data\CategoryWiseOverTime_{date_str}.xlsx"
)

SOURCE_SHEET = "Report_CategoryWiseOTWoven"


# =============================================================================
# 2. SOURCE COLUMN POSITIONS
# =============================================================================
# All positions are ZERO-BASED.
# =============================================================================

COLS = {
    "Factory": 0,
    "Type": 1,
    "Category": 2,
    "OTEOT": 3,

    # -------------------------------------------------------------------------
    # Month Total
    # -------------------------------------------------------------------------
    "MT_Emp": 4,
    "MT_Hour": 5,
    "MT_Amnt": 6,

    # -------------------------------------------------------------------------
    # Previous Day
    # -------------------------------------------------------------------------
    "PD_Emp": 7,
    "PD_Hour": 8,
    "PD_Amnt": 9,
}


# =============================================================================
# 3. NUMERIC COLUMNS
# =============================================================================

NUMERIC_COLS = [
    "MT_Emp",
    "MT_Hour",
    "MT_Amnt",
    "PD_Emp",
    "PD_Hour",
    "PD_Amnt",
]


# =============================================================================
# 4. KPI DEFINITIONS
# =============================================================================

KPI_SPEC = [
    (
        "Number of OT Employees",
        "PD_Emp",
        "OT",
    ),

    (
        "Number of Extra OT Employees",
        "PD_Emp",
        "EOT",
    ),

    (
        "Previous Days' OT Hours",
        "PD_Hour",
        "OT",
    ),

    (
        "Previous Days' Extra OT Hours",
        "PD_Hour",
        "EOT",
    ),

    (
        "Monthly Cumulative OT Hours",
        "MT_Hour",
        "OT",
    ),

    (
        "Monthly Cumultiave Extra OT Hours",
        "MT_Hour",
        "EOT",
    ),

    (
        "Monthly OT Expense",
        "MT_Amnt",
        "OT",
    ),

    (
        "Monthly Extra OT Expense",
        "MT_Amnt",
        "EOT",
    ),
]


# =============================================================================
# 5. KPI COLUMN ORDER
# =============================================================================

REPORT_KPI_COLS = [
    "Number of OT Employees",
    "Number of Extra OT Employees",
    "Previous Days' OT Hours",
    "Previous Days' Extra OT Hours",
    "Monthly Cumulative OT Hours",
    "Monthly Cumultiave Extra OT Hours",
    "Monthly OT Expense",
    "Monthly Extra OT Expense",
]


# =============================================================================
# 6. FINAL MANAGEMENT REPORT COLUMN ORDER
# =============================================================================
# Two requested blank columns:
#
#   1. OT and EOT
#      -> before Factory
#
#   2. Total Employees Present
#      -> immediately after Factory
# =============================================================================

REPORT_COLUMNS = [
    "OT and EOT",
    "Factory",
    "Total Employees Present",
    *REPORT_KPI_COLS,
]


# =============================================================================
# 7. SOURCE SUBTOTAL / TOTAL ROW DEFINITIONS
# =============================================================================

_AGG_CATEGORY_KEYS = {
    "total",
    "factot",
    "facteot",
}


_AGG_FACTORY_KEYS = {
    "totalot",
    "totaleot",
    "grandtotal",
}


# =============================================================================
# 8. TEXT NORMALIZATION HELPER
# =============================================================================

def _norm_key(value):
    """
    Convert a text value into a normalized alphabetic comparison key.

    Examples
    --------
    'Fact OT'      -> 'factot'
    'Grand Total'  -> 'grandtotal'
    'Woven'        -> 'woven'
    ' Extra OT '   -> 'extraot'
    """

    return re.sub(
        r"[^a-z]",
        "",
        str(value).lower(),
    )


# =============================================================================
# 9. TEXT CLEANING HELPER
# =============================================================================

def _clean_text(value):
    """
    Normalize whitespace and safely handle blank values.
    """

    if pd.isna(value):
        return np.nan

    value = str(value)

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value if value else np.nan


# =============================================================================
# 10. DATE INFORMATION EXTRACTOR
# =============================================================================

def _extract_report_dates(oteot):
    """
    Extract date labels from row 0 of the cleaned `oteot` DataFrame.

    Expected date format:
        DD-MMM-YYYY

    Example:
        23-Sep-2026
    """

    date_labels = []

    for value in oteot.iloc[0]:

        if not isinstance(value, str):
            continue

        value = value.strip()

        if re.match(
            r"^\d{2}-[A-Za-z]{3}-\d{4}$",
            value,
        ):
            date_labels.append(value)

    if not date_labels:
        return {}

    parsed_dates = pd.to_datetime(
        date_labels,
        format="%d-%b-%Y",
        errors="coerce",
    )

    parsed_dates = parsed_dates[
        ~parsed_dates.isna()
    ]

    if len(parsed_dates) == 0:
        return {}

    parsed_dates = sorted(parsed_dates)

    return {
        "Previous Day": parsed_dates[-1].strftime(
            "%d-%b-%Y"
        ),

        "Month Total Period": (
            f"{parsed_dates[0]:%d-%b-%Y} "
            f"to "
            f"{parsed_dates[-1]:%d-%b-%Y}"
        ),
    }


# =============================================================================
# 11. MAIN OT MANAGEMENT REPORT FUNCTION
# =============================================================================

def build_ot_management_report(oteot):
    """
    Build an OT / Extra-OT management report from cleaned `oteot`.

    Parameters
    ----------
    oteot : pandas.DataFrame
        Cleaned Category Wise OT report.

    Returns
    -------
    management_report : pandas.DataFrame
        Factory-wise management report with exactly the requested columns.

    report_info : dict
        Reporting period information.

    detail_kpi : pandas.DataFrame
        Factory + Type + Category KPI-level calculation.
    """

    # =========================================================================
    # STEP 1: Validate input object
    # =========================================================================

    if not isinstance(oteot, pd.DataFrame):

        raise TypeError(
            "`oteot` must be a pandas DataFrame."
        )


    # =========================================================================
    # STEP 2: Validate minimum rows and columns
    # =========================================================================

    required_rows = 3
    required_columns = max(COLS.values()) + 1

    if oteot.shape[0] < required_rows:

        raise ValueError(
            "`oteot` must contain at least 3 rows."
        )

    if oteot.shape[1] < required_columns:

        raise ValueError(
            f"`oteot` must contain at least "
            f"{required_columns} columns."
        )


    # =========================================================================
    # STEP 3: Extract reporting period information
    # =========================================================================

    report_info = _extract_report_dates(
        oteot
    )


    # =========================================================================
    # STEP 4: Extract actual data
    #
    # Row 0 -> reporting information
    # Row 1 -> actual headers
    # Row 2+ -> actual data
    # =========================================================================

    data = oteot.iloc[2:].copy()

    data = data.iloc[
        :,
        list(COLS.values()),
    ].copy()

    data.columns = list(
        COLS.keys()
    )


    # =========================================================================
    # STEP 5: Clean text columns
    # =========================================================================

    TEXT_COLS = [
        "Factory",
        "Type",
        "Category",
        "OTEOT",
    ]

    for column in TEXT_COLS:

        data[column] = data[column].apply(
            _clean_text
        )


    # =========================================================================
    # STEP 6: Remove completely invalid rows
    # =========================================================================

    data = data.dropna(
        subset=[
            "Factory",
            "Category",
            "OTEOT",
        ]
    ).reset_index(drop=True)


    # =========================================================================
    # STEP 7: Convert numeric columns
    # =========================================================================

    for column in NUMERIC_COLS:

        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        ).fillna(0)


    # =========================================================================
    # STEP 8: Remove source subtotal / total rows
    # =========================================================================

    category_key = data[
        "Category"
    ].apply(_norm_key)

    factory_key = data[
        "Factory"
    ].apply(_norm_key)

    is_aggregate = (
        category_key.isin(
            _AGG_CATEGORY_KEYS
        )
        |
        factory_key.isin(
            _AGG_FACTORY_KEYS
        )
    )

    real = data.loc[
        ~is_aggregate
    ].reset_index(drop=True)


    # =========================================================================
    # STEP 9: Validate remaining data
    # =========================================================================

    if real.empty:

        raise ValueError(
            "No valid OT / Extra-OT category rows "
            "remained after removing subtotal and "
            "grand-total rows."
        )


    # =========================================================================
    # STEP 10: Aggregate Factory + Type + Category + OT/EOT
    # =========================================================================

    grouped = (
        real
        .groupby(
            [
                "Factory",
                "Type",
                "Category",
                "OTEOT",
            ],
            sort=False,
            dropna=False,
        )[NUMERIC_COLS]
        .sum()
        .reset_index()
    )


    # =========================================================================
    # STEP 11: Create OT / EOT pivot
    # =========================================================================

    pivot = grouped.pivot_table(
        index=[
            "Factory",
            "Type",
            "Category",
        ],

        columns="OTEOT",

        values=NUMERIC_COLS,

        fill_value=0,

        aggfunc="sum",
    )


    # =========================================================================
    # STEP 12: Calculate detailed KPI
    # =========================================================================

    detail_kpi = pd.DataFrame(
        index=pivot.index
    )

    for (
        kpi_name,
        source_column,
        ot_eot,
    ) in KPI_SPEC:

        column_key = (
            source_column,
            ot_eot,
        )

        if column_key in pivot.columns:

            detail_kpi[kpi_name] = (
                pivot[column_key]
            )

        else:

            detail_kpi[kpi_name] = 0


    detail_kpi = (
        detail_kpi
        .reset_index()
    )


    # =========================================================================
    # STEP 13: Ensure KPI columns are numeric
    # =========================================================================

    for column in REPORT_KPI_COLS:

        detail_kpi[column] = pd.to_numeric(
            detail_kpi[column],
            errors="coerce",
        ).fillna(0)


    # =========================================================================
    # STEP 14: Factory + Type summary
    # =========================================================================

    type_summary = (
        detail_kpi
        .groupby(
            [
                "Factory",
                "Type",
            ],
            sort=False,
            dropna=False,
        )[REPORT_KPI_COLS]
        .sum()
        .reset_index()
    )


    # =========================================================================
    # STEP 15: Build factory-wise management report
    #
    # Rules:
    #
    # 1. If Woven exists for a factory:
    #
    #       MGL
    #       MGL-Knit
    #
    #    instead of:
    #
    #       MGL-Woven
    #       MGL-Knit
    #
    # 2. If a factory has only one Type:
    #
    #       Factory
    #
    #    is used directly.
    #
    # 3. If multiple types exist and Woven does not exist:
    #    first type becomes the base row.
    # =========================================================================

    rows = []

    BASE_TYPE_KEY = "woven"

    factory_order = list(
        dict.fromkeys(
            real["Factory"].tolist()
        )
    )


    for factory in factory_order:

        factory_types = (
            type_summary.loc[
                type_summary["Factory"] == factory,
                "Type",
            ]
            .tolist()
        )

        if not factory_types:
            continue


        # ---------------------------------------------------------------------
        # Find Woven type
        # ---------------------------------------------------------------------

        woven_type = next(
            (
                item
                for item in factory_types
                if _norm_key(item) == BASE_TYPE_KEY
            ),
            None,
        )


        # ---------------------------------------------------------------------
        # Determine base type
        # ---------------------------------------------------------------------

        if woven_type is not None:

            base_type = woven_type

        elif len(factory_types) == 1:

            base_type = factory_types[0]

        else:

            base_type = factory_types[0]


        # ---------------------------------------------------------------------
        # Get base type summary
        # ---------------------------------------------------------------------

        base_row = type_summary.loc[
            (
                type_summary["Factory"] == factory
            )
            &
            (
                type_summary["Type"] == base_type
            )
        ]


        if base_row.empty:
            continue


        base_values = (
            base_row.iloc[0]
            [REPORT_KPI_COLS]
            .to_dict()
        )


        rows.append(
            {
                "Factory": factory,
                **base_values,
            }
        )


        # ---------------------------------------------------------------------
        # Add additional types
        # ---------------------------------------------------------------------

        for extra_type in factory_types:

            if extra_type == base_type:
                continue


            extra_row = type_summary.loc[
                (
                    type_summary["Factory"] == factory
                )
                &
                (
                    type_summary["Type"] == extra_type
                )
            ]


            if extra_row.empty:
                continue


            extra_values = (
                extra_row.iloc[0]
                [REPORT_KPI_COLS]
                .to_dict()
            )


            rows.append(
                {
                    "Factory": (
                        f"{factory}-{extra_type}"
                    ),
                    **extra_values,
                }
            )


    # =========================================================================
    # STEP 16: Grand Total
    # =========================================================================

    total_row = {
        "Factory": "Total",
    }


    for column in REPORT_KPI_COLS:

        total_row[column] = (
            detail_kpi[column].sum()
        )


    rows.append(
        total_row
    )


    # =========================================================================
    # STEP 17: Create management report DataFrame
    # =========================================================================

    management_report = pd.DataFrame(
        rows,
        columns=[
            "Factory",
            *REPORT_KPI_COLS,
        ],
    )


    # =========================================================================
    # STEP 18: Ensure KPI columns are numeric
    # =========================================================================

    for column in REPORT_KPI_COLS:

        management_report[column] = pd.to_numeric(
            management_report[column],
            errors="coerce",
        ).fillna(0)


    # =========================================================================
    # STEP 19: Add requested blank columns
    #
    # Final structure:
    #
    #     OT and EOT
    #     Factory
    #     Total Employees Present
    #     KPI 1
    #     KPI 2
    #     ...
    #
    # These two columns intentionally contain blank values.
    # =========================================================================

    management_report.insert(
        0,
        "OT and EOT",
        "",
    )


    management_report.insert(
        2,
        "Total Employees Present",
        "",
    )


    # =========================================================================
    # STEP 20: Enforce final column order
    # =========================================================================

    management_report = management_report[
        REPORT_COLUMNS
    ].copy()


    # =========================================================================
    # STEP 21: Return
    # =========================================================================

    return (
        management_report,
        report_info,
        detail_kpi,
    )


# =============================================================================
# 12. LOAD CLEANED OT / EOT SOURCE
# =============================================================================
#
# `clean_category_wise_overtime()` should already be available from the
# previous cleaning cell/function.
# =============================================================================

if not INPUT_FILE.exists():

    raise FileNotFoundError(
        f"Input file was not found:\n\n"
        f"{INPUT_FILE.resolve()}\n\n"
        f"Expected filename pattern:\n"
        f"CategoryWiseOverTime_{date_str}.xlsx"
    )


oteot = clean_category_wise_overtime(
    input_file=str(INPUT_FILE),
    source_sheet=SOURCE_SHEET,
)


# =============================================================================
# 13. BUILD OT / EOT MANAGEMENT REPORT
# =============================================================================

oteot_report, report_info, detail_kpi = (
    build_ot_management_report(
        oteot
    )
)


# =============================================================================
# 14. DISPLAY REPORT INFORMATION
# =============================================================================

print("=" * 80)
print("OT / EOT MANAGEMENT REPORT")
print("=" * 80)

print(
    f"Input File : {INPUT_FILE}"
)

if report_info:

    print(
        f"Previous Day : "
        f"{report_info.get('Previous Day', 'N/A')}"
    )

    print(
        f"Month Total Period : "
        f"{report_info.get('Month Total Period', 'N/A')}"
    )

print(
    f"Rows : {len(oteot_report)}"
)

print(
    f"Columns : {len(oteot_report.columns)}"
)

print("=" * 80)


# =============================================================================
# 15. FINAL REPORT
# =============================================================================

display(
    oteot_report
)

OT / EOT MANAGEMENT REPORT
Input File : ..\Data\CategoryWiseOverTime_25092026.xlsx
Previous Day : 23-Sep-2026
Month Total Period : 01-Sep-2026 to 23-Sep-2026
Rows : 7
Columns : 11


,OT and EOT,Factory,Total Employees Present,Number of OT Employees,Number of Extra OT Employees,Previous Days' OT Hours,Previous Days' Extra OT Hours,Monthly Cumulative OT Hours,Monthly Cumultiave Extra OT Hours,Monthly OT Expense,Monthly Extra OT Expense
0,,MGL,,833.0,214.0,1658.5,467.0,31141.5,14481.5,2564833.26,1194652.95
1,,MGL-Knit,,264.0,20.0,518.5,64.0,17031.0,11937.0,1372749.99,964141.58
2,,MGNSL,,804.0,21.0,1595.0,61.0,32746.5,8840.5,2650018.29,716384.22
3,,MGSL,,466.0,1.0,911.5,1.0,25607.0,9025.0,2155418.90,758057.42
4,,MGSLE,,150.0,9.0,241.5,26.0,10810.0,4794.5,842328.07,373780.55
5,,TMLS,,29.0,24.0,55.5,30.5,1286.5,683.0,107508.64,58604.18
6,,Total,,2546.0,289.0,4980.5,649.5,118622.5,49761.5,9692857.15,4065620.90


## Data Process End Here

## 3. Build and inspect the local master DataFrame

In [69]:
reports = [
    ("Report 1", less_production),
    ("Report 2", fabric_delay_df),
    ("Report 3", line_wise_earning),
    ("Report 4", oteot_report)
]

print("Reports:", [name for name, _ in reports])


Reports: ['Report 1', 'Report 2', 'Report 3', 'Report 4']


In [70]:
master_df = build_master_dataframe(reports)
display(master_df)


,0,1,2,3,4,5,6,7,8,9,10
0,Report 1,,,,,,,,,,
1,Less Production,Factory,Line,Buyer,Style,Day Target Qty,Achieve Qty,Percentage,,,
2,,MGL,Line 01,STREETONE,346497/3413355,582,400,68.73,,,
3,,MGL,Line 02,US POLO,AW26-FM-SKT-002,364,120,32.97,,,
4,Yesterday,MGL,Line 06,NEXT,Y63406 - GIRLS PANT,"1,500","1,000",66.67,,,
...,...,...,...,...,...,...,...,...,...,...,...
96,,MGNSL,,804.0,21.0,1595.0,61.0,32746.5,8840.5,2650018.29,716384.22
97,,MGSL,,466.0,1.0,911.5,1.0,25607.0,9025.0,2155418.9,758057.42
98,,MGSLE,,150.0,9.0,241.5,26.0,10810.0,4794.5,842328.07,373780.55
99,,TMLS,,29.0,24.0,55.5,30.5,1286.5,683.0,107508.64,58604.18


## 4. Previous-day target

The system always uses Bangladesh time and writes yesterday into a sheet named `DD_MMM_YYYY`. Example: `2026-09-20` → `20_Sep_2026`.

In [71]:
previous_day = get_previous_report_date()
target_sheet = format_sheet_name(previous_day)

print("Previous report date:", previous_day.isoformat())
print("Target sheet name   :", target_sheet)


Previous report date: 2026-09-25
Target sheet name   : 25_Sep_2026


## 5. Build strict JSON-safe payload

In [72]:
payload = build_payload(
    reports=reports,
    report_date=previous_day,
)

json.dumps(payload, allow_nan=False)

print("Payload version:", payload["version"])
print("Run ID         :", payload["run_id"])
print("Report date    :", payload["report_date"])
print("Report count   :", len(payload["reports"]))
print("Strict JSON    : PASS")


Payload version: 1
Run ID         : 507592bc1dfe4b0ea2776da60bedf0c3
Report date    : 2026-09-25
Report count   : 4
Strict JSON    : PASS


## 6. Upload once

In [73]:
result = upload_to_google_apps_script(payload)

if result.get("success"):
    print("UPLOAD SUCCESS")
    print("Sheet        :", result.get("sheet"))
    print("Reports      :", result.get("report_count"))
    print("Rows written :", result.get("rows_written"))
    print("Run ID       :", result.get("run_id"))
else:
    print("UPLOAD FAILED")
    print("Reason       :", result.get("error"))
    raise RuntimeError(result.get("error", "Unknown upload error"))


UPLOAD SUCCESS
Sheet        : 25_Sep_2026
Reports      : 4
Rows written : 101
Run ID       : 507592bc1dfe4b0ea2776da60bedf0c3


## 7. Local data-type test

Use this once after installation to verify NaN, NaT, NumPy values, booleans, dates, and datetimes serialize correctly.

In [74]:
import datetime as dt
import numpy as np

messy = pd.DataFrame({
    "text": ["ok", None, "00123"],
    "float": [1.5, np.nan, np.inf],
    "int": np.array([1, 2, 3], dtype=np.int64),
    "date": [pd.Timestamp("2026-09-20"), pd.NaT, pd.Timestamp("2026-09-20 14:30:05")],
    "flag": np.array([True, False, True]),
    "pydate": [dt.date(2026, 9, 20), None, dt.datetime(2026, 9, 20, 8, 0, 0)],
})

check_payload = build_payload([("Messy Report", messy)], previous_day)
json.dumps(check_payload, allow_nan=False)
print(json.dumps(check_payload, indent=2, ensure_ascii=False))
print("Strict JSON: PASS")


{
  "version": 1,
  "run_id": "6bc4de50d6324888a0dcdc7412c8ab11",
  "report_date": "2026-09-25",
  "reports": [
    {
      "name": "Messy Report",
      "headers": [
        "text",
        "float",
        "int",
        "date",
        "flag",
        "pydate"
      ],
      "rows": [
        [
          "ok",
          1.5,
          1,
          "2026-09-20",
          true,
          "2026-09-20"
        ],
        [
          "",
          "",
          2,
          "",
          false,
          ""
        ],
        [
          "00123",
          "",
          3,
          "2026-09-20 14:30:05",
          true,
          "2026-09-20 08:00:00"
        ]
      ]
    }
  ]
}
Strict JSON: PASS
